# On the Efficacy of Shorting Corporate Bonds as a Tail Risk Hedging Solution

## Strategy Description
This notebook implements the strategy from the paper "On the Efficacy of Shorting Corporate Bonds as a Tail Risk Hedging Solution" by Travis Cable, Amir Mani, Wei Qi, Georgios Sotiropoulos, and Yiyuan Xiong. The strategy involves shorting investment-grade (IG) corporate bonds during periods of market drawdowns to hedge against downside risk. The strategy uses three signals: Momentum, Liquidity, and Credit, to determine when to enter and exit short positions in IG ETFs.

**Paper Citation:**
Cable, T., Mani, A., Qi, W., Sotiropoulos, G., & Xiong, Y. (2025). On the Efficacy of Shorting Corporate Bonds as a Tail Risk Hedging Solution. *arXiv preprint arXiv:2504.06289*.

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 — Trading Context & Objectives

In [ ]:
# Configuration
UNIVERSE = ['AAPL', 'MSFT']
ETF = 'LQD'  # Example IG ETF
MOMENTUM_WINDOW = 20
LIQUIDITY_WINDOW = 20
CREDIT_WINDOW = 20
POSITION_SIZE = 0.1  # 10% of portfolio

# Hypothesis
# The strategy hypothesizes that shorting IG corporate bonds during market drawdowns can effectively hedge against downside risk.

## Phase 2 — Data Download & Feature Computation

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Download data
data = yf.download(UNIVERSE + [ETF], start='2020-01-01', end='2023-01-01')

# Compute features
data['Momentum'] = data['Close'].pct_change(MOMENTUM_WINDOW)
data['Liquidity'] = data['Volume'].rolling(LIQUIDITY_WINDOW).mean()
data['Credit'] = data['Close'].pct_change(CREDIT_WINDOW)

# Cross-sectional normalization
data['Momentum'] = data['Momentum'].rank(axis=1, pct=True)
data['Liquidity'] = data['Liquidity'].rank(axis=1, pct=True)
data['Credit'] = data['Credit'].rank(axis=1, pct=True)

## Phase 3 — Signal Generation & Portfolio Construction

In [ ]:
# Signal generation
data['Signal'] = (data['Momentum'] + data['Liquidity'] + data['Credit']) / 3

# Position sizing
data['Position'] = np.where(data['Signal'] > 0.5, POSITION_SIZE, -POSITION_SIZE)

# Portfolio construction
data['Portfolio'] = data['Position'].shift(1) * data['Close'].pct_change()

## Phase 4 — Vectorized Backtest

In [ ]:
# Vectorized backtest
data['Cumulative_Return'] = (1 + data['Portfolio']).cumprod()

# Plot equity curve
import matplotlib.pyplot as plt

plt.plot(data['Cumulative_Return'])
plt.title('Equity Curve')
plt.show()

## Phase 5 — Performance Metrics

In [ ]:
from scipy.stats import norm

# Performance metrics
annual_return = data['Portfolio'].mean() * 252
annual_volatility = data['Portfolio'].std() * np.sqrt(252)
sharpe_ratio = annual_return / annual_volatility
sortino_ratio = annual_return / data['Portfolio'][data['Portfolio'] < 0].std() * np.sqrt(252)
max_drawdown = (data['Cumulative_Return'].cummax() - data['Cumulative_Return']).max()
calmar_ratio = annual_return / max_drawdown

print(f'Annual Return: {annual_return:.2%}')
print(f'Annual Volatility: {annual_volatility:.2%}')
print(f'Sharpe Ratio: {sharpe_ratio:.2f}')
print(f'Sortino Ratio: {sortino_ratio:.2f}')
print(f'Max Drawdown: {max_drawdown:.2%}')
print(f'Calmar Ratio: {calmar_ratio:.2f}')

## Phase 6 — Monitoring Stub

In [ ]:
# Monitoring stub
def monitor_daily_pnl(data):
    daily_pnl = data['Portfolio'].iloc[-1] * data['Close'].iloc[-1]
    current_positions = data['Position'].iloc[-1]
    print(f'Daily P&L: {daily_pnl:.2f}')
    print(f'Current Positions: {current_positions}')

# Example usage
monitor_daily_pnl(data)